# Format

In [73]:
import numpy as np
import os
import json
import pandas as pd

In [74]:
# DP3, 2024 data
data_df = pd.read_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2024\PUMA\1_yr_ACS_2024_PUMA_NYC_DP03.csv")
data_df

,DP03_0001E,DP03_0001EA,DP03_0001M,DP03_0001MA,DP03_0001PE,DP03_0001PEA,DP03_0001PM,DP03_0001PMA,DP03_0002E,DP03_0002EA,...,DP03_0137MA,DP03_0137PE,DP03_0137PEA,DP03_0137PM,DP03_0137PMA,GEO_ID,NAME,state,public use microdata area,vintage
0,144345,NaN,9269,NaN,144345,NaN,-888888888,NaN,84740,NaN,...,NaN,31.7,NaN,5.0,NaN,NaN,NYC-Manhattan Community District 3--Lower East...,36,4103,2024
1,115236,NaN,9901,NaN,115236,NaN,-888888888,NaN,88148,NaN,...,NaN,15.5,NaN,2.9,NaN,NaN,NYC-Manhattan Community District 4--Chelsea & ...,36,4104,2024
2,202107,NaN,12109,NaN,202107,NaN,-888888888,NaN,134341,NaN,...,NaN,17.9,NaN,3.4,NaN,NaN,NYC-Manhattan Community District 7--Upper West...,36,4107,2024
3,193797,NaN,10319,NaN,193797,NaN,-888888888,NaN,137634,NaN,...,NaN,11.3,NaN,2.4,NaN,NaN,NYC-Manhattan Community District 8--Upper East...,36,4108,2024
4,102122,NaN,8252,NaN,102122,NaN,-888888888,NaN,62957,NaN,...,NaN,33.8,NaN,7.5,NaN,NaN,NYC-Manhattan Community District 9--Morningsid...,36,4109,2024
5,105499,NaN,10057,NaN,105499,NaN,-888888888,NaN,69618,NaN,...,NaN,31.1,NaN,5.4,NaN,NaN,NYC-Manhattan Community District 10--Harlem PU...,36,4110,2024
6,111508,NaN,10100,NaN,111508,NaN,-888888888,NaN,63372,NaN,...,NaN,42.6,NaN,5.1,NaN,NaN,NYC-Manhattan Community District 11--East Harl...,36,4111,2024
7,155889,NaN,10361,NaN,155889,NaN,-888888888,NaN,98623,NaN,...,NaN,27.8,NaN,4.6,NaN,NaN,NYC-Manhattan Community District 12--Washingto...,36,4112,2024
8,142503,NaN,10560,NaN,142503,NaN,-888888888,NaN,108486,NaN,...,NaN,9.3,NaN,2.7,NaN,NaN,NYC-Manhattan Community Districts 1 & 2--Finan...,36,4121,2024
9,191133,NaN,12244,NaN,191133,NaN,-888888888,NaN,143202,NaN,...,NaN,14.8,NaN,3.1,NaN,NaN,NYC-Manhattan Community Districts 5 & 6--Midto...,36,4165,2024


In [75]:
# labels
with open(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\HTM Projects\ACS_Warehouse\docs\dp_variables_2024.json", 'r') as f:
    data = json.load(f)

meta_df = (
    pd.DataFrame.from_dict(data["variables"], orient="index")
      .reset_index()
      .rename(columns={"index": "variable"})
)
meta_df

,variable,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
0,for,Census API FIPS 'for' clause,Census API Geography Specification,fips-for,N/A,0,True,NaN,NaN,NaN
1,in,Census API FIPS 'in' clause,Census API Geography Specification,fips-in,N/A,0,True,NaN,NaN,NaN
2,ucgid,Uniform Census Geography Identifier clause,Census API Geography Specification,ucgid,N/A,0,True,True,NaN,NaN
3,DP02_0126E,Estimate!!ANCESTRY!!Total population!!Arab,Selected Social Characteristics in the United ...,int,DP02,0,NaN,NaN,"DP02_0126EA,DP02_0126M,DP02_0126MA",NaN
4,DP05_0050PE,Percent!!RACE!!Total population!!One race!!Bla...,ACS Demographic and Housing Estimates,float,DP05,0,NaN,NaN,"DP05_0050PEA,DP05_0050PM,DP05_0050PMA",NaN
...,...,...,...,...,...,...,...,...,...,...
1412,DP03_0039PE,Percent!!INDUSTRY!!Civilian employed populatio...,Selected Economic Characteristics,float,DP03,0,NaN,NaN,"DP03_0039PEA,DP03_0039PM,DP03_0039PMA",NaN
1413,DP02_0098E,Estimate!!YEAR OF ENTRY!!Population born outsi...,Selected Social Characteristics in the United ...,int,DP02,0,NaN,NaN,"DP02_0098EA,DP02_0098M,DP02_0098MA",NaN
1414,DP04_0095PE,Percent!!SELECTED MONTHLY OWNER COSTS (SMOC)!!...,Selected Housing Characteristics,float,DP04,0,NaN,NaN,"DP04_0095PEA,DP04_0095PM,DP04_0095PMA",NaN
1415,DP02_0036PE,Percent!!MARITAL STATUS!!Females 15 years and ...,Selected Social Characteristics in the United ...,float,DP02,0,NaN,NaN,"DP02_0036PEA,DP02_0036PM,DP02_0036PMA",NaN


In [ ]:
# Extract parameters for function and file name
sample = '1_yr' # or 5_yr
vintage = '2024'
geography = 'PUMA'

In [77]:
# Define Function to format tables
def acs_dp_to_wide(data_df:pd.DataFrame, meta_df:pd.DataFrame, sample:str, vintage:int) -> tuple:
    """
    Function is designed to work with data profile coming from census API requests
    
    Args:
        df_api: Data Profile DataFrame 
        df_meta: Metadata DataFrame
        sample: 1_yr or 5_yr
        vintage: survey year (or last year in 5_yr samples)
    
    Returns:
        pd.DataFrame: Wide format table (variables in rows, geographies in columns) with data profile data

    """
    # PREPARE DATA
    # Remove unnecessary columns from data df
    var_df = data_df.loc[:, ~data_df.columns.str.endswith(('EA', 'MA', 'PEA'))].copy()      # drop annotation columns
    var_df = var_df.drop(columns = ['GEO_ID', 'state', 'vintage', 'NAME'])                  # drop geo_id column (it's empty), state (non-necessary), vintage (useful but makes transpose more difficult),NAME (useful, but messes up the column type)
    
    # Recode missing values
    var_df = var_df.replace([-888888888, -999999999], np.nan)
    
    # Transpose dataframe
    wide_df = var_df.transpose()
    headers = wide_df.iloc[-1].astype(int)                                          # type int to remove decimal point
    wide_df.columns = headers
    wide_df = wide_df[:-1].copy()                                                   # Remove row with headers
    wide_df = wide_df.add_prefix('puma_')
    wide_df = wide_df.reset_index().rename(columns={'index': 'variable'})           # reset index

    # Reincorporate vintage
    wide_df['vintage'] = vintage

    # Add sample indicator
    wide_df['sample'] = sample
    
    # Get variable base name
    wide_df['var_name_base'] = wide_df['variable'].str[0:9]

    # Name qualifier / variable type
    wide_df['var_type'] = wide_df['variable'].str.extract(r'\d+([A-Z]+)', expand=False)
    
    # Change description in var type
    wide_df['var_type'] = np.where(wide_df['var_type'] == 'E', 'Estimate',
                                np.where(wide_df['var_type'] == 'M', 'Estimate MOE',
                                            np.where(wide_df['var_type'] == 'PE', 'Percentage',
                                                     np.where(wide_df['var_type'] == 'PM', 'Percentage MOE', np.nan))))
    
    # PREPARE META DATA
    # Sort values
    meta_df = meta_df.sort_values(by = 'variable')
    
    # Keep only variables in groups DO02 to DP05
    meta_var_df = meta_df[meta_df['group'].isin(['DP02', 'DP03', 'DP04', 'DP05'])].copy()
    
    # Variable base name
    meta_var_df['var_name_base'] = meta_var_df['variable'].str[0:9]

    # Name qualifier / var type
    meta_var_df['var_type'] = meta_var_df['variable'].str.extract(r'\d+([A-Z]+)', expand=False)
    
    # Gen new label column without 'Estimate' or 'Percent' prefix
    meta_var_df['base_label'] = meta_var_df['label'].replace('Estimate|Percent', '', regex=True)
    
    # Keep only the columns we need
    meta_labels_df = meta_var_df[['var_name_base', 'base_label', 'group']].copy()
    
    # Drop duplicates
    meta_labels_df = meta_labels_df.drop_duplicates()
    
    # Label format: Remove initial '!!'
    meta_labels_df['base_label'] = meta_labels_df['base_label'].str.lstrip('!!')

    # Label format: Add a column for the topic of the variable (could be useful when applying filters in Excel)
    meta_labels_df['var_topic'] = meta_labels_df['base_label'].str.split('!!').str[0]

    # Label format: Replace '!!' with ' - '
    meta_labels_df['base_label'] = meta_labels_df['base_label'].str.replace('!!', ' - ', regex=True)

    
    
    # MERGE DATA WITH LABELS
    # Merge
    final_df = pd.merge(wide_df, meta_labels_df, how = 'left', left_on='var_name_base', right_on='var_name_base')
    
    # FORMAT FINAL DATA FRAME
    # Drop unnecessary columns
    final_df = final_df.drop(columns = ['var_name_base'])               # Used only for merge
    # rename columns
    final_df = final_df.rename(columns = {'base_label' : 'var_label', 'group' : 'var_group'})
    # sort columns
    final_df = final_df.sort_index(axis = 1, ascending = False)
    
    # GEO CODES
    geo_labels = data_df[['public use microdata area', 'state', 'NAME']]
    geo_labels = geo_labels.rename(columns={'public use microdata area' : 'puma_code', 'state' : 'state_code', 'NAME' : 'puma_name'} )
    
    return(final_df, geo_labels)  

In [78]:
dp_wide, geocodes = acs_dp_to_wide(data_df=data_df, meta_df=meta_df, sample='1_yr', vintage=2024)

In [79]:
dp_wide

,vintage,variable,var_type,var_topic,var_label,var_group,sample,puma_4503,puma_4502,puma_4501,...,puma_4165,puma_4121,puma_4112,puma_4111,puma_4110,puma_4109,puma_4108,puma_4107,puma_4104,puma_4103
0,2024,DP03_0001E,Estimate,EMPLOYMENT STATUS,EMPLOYMENT STATUS - Population 16 years and over,DP03,1_yr,138293.0,115611.0,152902.0,...,191133.0,142503.0,155889.0,111508.0,105499.0,102122.0,193797.0,202107.0,115236.0,144345.0
1,2024,DP03_0001M,Estimate MOE,EMPLOYMENT STATUS,EMPLOYMENT STATUS - Population 16 years and over,DP03,1_yr,9496.0,7868.0,7711.0,...,12244.0,10560.0,10361.0,10100.0,10057.0,8252.0,10319.0,12109.0,9901.0,9269.0
2,2024,DP03_0001PE,Percentage,EMPLOYMENT STATUS,EMPLOYMENT STATUS - Population 16 years and over,DP03,1_yr,138293.0,115611.0,152902.0,...,191133.0,142503.0,155889.0,111508.0,105499.0,102122.0,193797.0,202107.0,115236.0,144345.0
3,2024,DP03_0001PM,Percentage MOE,EMPLOYMENT STATUS,EMPLOYMENT STATUS - Population 16 years and over,DP03,1_yr,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024,DP03_0002E,Estimate,EMPLOYMENT STATUS,EMPLOYMENT STATUS - Population 16 years and ov...,DP03,1_yr,88797.0,68769.0,91362.0,...,143202.0,108486.0,98623.0,63372.0,69618.0,62957.0,137634.0,134341.0,88148.0,84740.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
543,2024,DP03_0136PM,Percentage MOE,PERCENTAGE OF FAMILIES AND PEOPLE WHOSE INCOME...,PERCENTAGE OF FAMILIES AND PEOPLE WHOSE INCOME...,DP03,1_yr,2.3,2.7,3.5,...,3.0,2.5,4.7,8.4,7.4,8.1,1.1,2.0,4.6,5.8
544,2024,DP03_0137E,Estimate,PERCENTAGE OF FAMILIES AND PEOPLE WHOSE INCOME...,PERCENTAGE OF FAMILIES AND PEOPLE WHOSE INCOME...,DP03,1_yr,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
545,2024,DP03_0137M,Estimate MOE,PERCENTAGE OF FAMILIES AND PEOPLE WHOSE INCOME...,PERCENTAGE OF FAMILIES AND PEOPLE WHOSE INCOME...,DP03,1_yr,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
546,2024,DP03_0137PE,Percentage,PERCENTAGE OF FAMILIES AND PEOPLE WHOSE INCOME...,PERCENTAGE OF FAMILIES AND PEOPLE WHOSE INCOME...,DP03,1_yr,19.0,28.6,32.3,...,14.8,9.3,27.8,42.6,31.1,33.8,11.3,17.9,15.5,31.7


In [80]:
sample = '1_yr'
vintage = '2024'
geography = 'PUMA'
print(fr'C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\{sample}_sample\{vintage}\{geography}')

C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2024\PUMA


In [81]:
# Save data

directory = r'C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2024\PUMA'
file_name = '1_yr_ACS_2024_PUMA_NYC_DP03_ECONOMIC_CHAR.xlsx'

# Ensure the folder exists (create if missing)
os.makedirs(directory, exist_ok=True)


full_path = os.path.join(directory, file_name)

with pd.ExcelWriter(full_path) as writer:
    dp_wide.to_excel(writer, sheet_name = 'data', index = False)
    geocodes.to_excel(writer, sheet_name = 'geo_labels', index = False)